# Taller 3

Descargamos las librerías

In [ ]:
!apt install libgraphviz-dev
!pip install pygraphviz
!pip install pgmpy

In [ ]:
from itertools import combinations

import networkx as nx
import pandas as pd
import numpy as np

from pgmpy.estimators import PC, HillClimbSearch
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Image
import matplotlib.image as mpimg

## 0. Lectura de los datos

Usaremos datos de la web de [Fama-French](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). Usaremos los datos de los factores de Fama French, así como los datos de las 5 principales industrias. En total tenemos las siguientes variables en nuestro dataset:

- **Mkt-RF**: Variable del mercado descontando el activo libre de riesgo
- **SMB**: Variable del factor tamaño de las empresas
- **HML**: Variable del factor de capitalización de las empresas
- **RF**: Variable del activo libre de riesgo
- **Mom**: Variable del factor momentum
- **Cnsmr**: Variable del sector de consumo
- **Manuf**: Variable del sector manufacturero
- **HiTec**: Variable del sector tecnológico
- **Hlth**: Variable del sector de la salud
- **Other**: Variable de cualquier otro sector, es decir, el resto del mercado

In [ ]:
# Leemos los datos
data_FF = pd.read_csv('datos_Fama_French.csv', header = 0)
data_momentum = pd.read_csv('datos_momentum.csv', header = 0)
data_industries = pd.read_csv('datos_industrias_5.csv', header = 0)

# Marcamos la columna Date como índice para tener ordenadas las observaciones y hacer las concatenaciones (merge)
data_FF['Date'] = pd.to_datetime(data_FF['Date'], format='%Y%m%d')
data_momentum['Date'] = pd.to_datetime(data_momentum['Date'], format='%Y%m%d')
data_industries['Date'] = pd.to_datetime(data_industries['Date'], format='%Y%m%d')

# Marcamos la fecha como índice
data_FF = data_FF.set_index('Date')
data_momentum = data_momentum.set_index('Date')
data_industries = data_industries.set_index('Date')

# Concatenamos los tres datasets. Seleccionamos los datos desde 1927 porque no hay nulos
df = pd.concat([data_FF, data_momentum, data_industries], axis = 1).loc['1927-01-01':]
df.columns = [col.replace('-', '') for col in df.columns]
samples = df

In [ ]:
samples

# 1. Cálculo de la matriz de correlaciones

Podemos apreciar como los **factores Fama-French están descorrelacionados** con respecto a todas las variables.

La única **excepción** es la variable **Mkt-RF** que, como era de esperar, está altamente **correlacionada con los factores de las cinco industrias**.

In [ ]:
# Obtenemos la matriz de correlaciones
df_corr = df.corr()

# La mostramos en un gráfico
plt.figure(figsize = (12,8))
sns.heatmap(df_corr, vmin = -1, vmax = 1, cmap = 'coolwarm', annot = True, fmt=".2f", linewidths=0.5)
plt.title('Matriz de Correlación de Factores Financieros')

# 2. Cálculo de la matriz direccional

Calculamos la matriz direccional. Esta matriz asigna las **relaciones de causalidad** entre variables. Las relaciones pueden ser en un sentido o en los dos entre dos variables o nodos. Es decir, puede haber una relación de causalidad $A → B$ pero también $B → A$.

La celda $(i,j)$ de la matriz direccional muestra la variable $i$ desde la que sale la arista hasta la variable $j$. Todas las celdas pintadas en amarillo implicarían esta relación de causalidad entre las variables $i$ y $j$.

Por ejemplo, destaca la **relación de causalidad** entre las variables **Mkt-Rf y Hlth** de la matriz direccional del gráfico cuyo título es "**Matrices direccionales dirigidas**". En este caso sería la celda con la fila Hlth y la columna Mkt-Rf.

- La relación $Hlth → Mkt-Rf$ presenta una causalidad significativa porque la celda (Mkt-Rf, Hlth) está pintada de amarillo.
- La relación de causalidad $Mkt-Rf → Hlth$ no porque la celda (Mkt-Rf, Hlth) no está pintada de amarillo.

In [ ]:
# Modelo PC
est_PC = PC(data=samples)
estimated_model_PC = est_PC.estimate(variant="stable", max_cond_vars=2)

# Modelo HC
est_HC = HillClimbSearch(data=samples)
estimated_model_HC = est_HC.estimate()

In [ ]:
# Sin flechas direccionales
nodes_PC1 = estimated_model_PC.nodes()
est_adj_PC1 = nx.to_numpy_array(estimated_model_PC.to_undirected(), nodelist=nodes_PC1, weight=None)

nodes_HC1 = estimated_model_HC.nodes()
est_adj_HC1 = nx.to_numpy_array(estimated_model_HC.to_undirected(), nodelist=nodes_HC1, weight=None)

# Creamos una figura con 1 fila y 2 columnas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9))

# Gráfico 1: Algoritmo PC
im1 = ax1.imshow(est_adj_PC1)
ax1.set_title("Matriz direccional: Algoritmo PC")
ax1.set_xticks(range(len(df.columns)))
ax1.set_xticklabels(df.columns, rotation=45)
ax1.set_yticks(range(len(df.columns)))
ax1.set_yticklabels(df.columns)

# Gráfico 2: Algoritmo HC (Hill Climbing)
im2 = ax2.imshow(est_adj_HC1)
ax2.set_title("Matriz direccional: Algoritmo HC")
ax2.set_xticks(range(len(df.columns)))
ax2.set_xticklabels(df.columns, rotation=45)
ax2.set_yticks(range(len(df.columns)))
ax2.set_yticklabels(df.columns)

plt.tight_layout()
fig.suptitle('Matrices direccionales no dirigidas', fontsize=16)
plt.show()

In [ ]:
# Con flechas direccionales
nodes_PC2 = estimated_model_PC.nodes()
est_adj_PC2 = nx.to_numpy_array(estimated_model_PC.to_directed(), nodelist=list(df.columns), weight=None)

nodes_HC2 = estimated_model_HC.nodes()
est_adj_HC2 = nx.to_numpy_array(estimated_model_HC.to_directed(), nodelist=list(df.columns), weight=None)

# Creamos una figura con 1 fila y 2 columnas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9))

# Gráfico 1: Algoritmo PC
im1 = ax1.imshow(est_adj_PC2)
ax1.set_title("Matriz direccional: Algoritmo PC")
ax1.set_xticks(range(len(df.columns)))
ax1.set_xticklabels(df.columns, rotation=45)
ax1.set_yticks(range(len(df.columns)))
ax1.set_yticklabels(df.columns)

# Gráfico 2: Algoritmo HC (Hill Climbing)
im2 = ax2.imshow(est_adj_HC2)
ax2.set_title("Matriz direccional: Algoritmo HC")
ax2.set_xticks(range(len(df.columns)))
ax2.set_xticklabels(df.columns, rotation=45)
ax2.set_yticks(range(len(df.columns)))
ax2.set_yticklabels(df.columns)

plt.tight_layout()
fig.suptitle('Matrices direccionales dirigidas', fontsize = 16)
plt.show()

# 3. Grafos Acíclicos Dirigidos (DAG)

A continuación se muestran las relaciones de causalidad entre las variables del dataset utilizado. El primer gráfico muestra la relación entre cada par de variables, si existe, y su dirección (**grafo dirigido**) mientras que el segundo muestra simplemente los casos en los que existe una relación causal significativa independientemente de su dirección (**grafo no dirigido**).

Siguiendo con el ejemplo del apartado anterior, en el **grafo dirigido**:

- La relación **$Hlth → Mkt-Rf$** presenta una **causalidad significativa** porque hay un arista que inicia en el nodo Hlth y finaliza en el nodo Mkt-Rf.
- La relación de causalidad **$Mkt-Rf → Hlth$ no existe una causalidad significativa** porque no hay ninguna arista que conecte los nodos Mkt-Rf y Hlth.

In [ ]:
# Diccionario para modificar los nombres de las matrices direccionales y ajustarlas a los nombres de las variables
mapping = {i: col for i, col in enumerate(df.columns)}

# Grafo Dirigido (Con flechas)
G_dir = nx.DiGraph(estimated_model_PC.to_directed())
G_dir = nx.relabel_nodes(G_dir, mapping)
pos_dir = nx.spring_layout(G_dir, k=2, seed=42)

# Grafo No Dirigido (Sin flechas)
G_undir = nx.Graph(estimated_model_PC.to_undirected())
G_undir = nx.relabel_nodes(G_undir, mapping)
pos_undir = nx.spring_layout(G_undir, k=2, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))
node_params = {"with_labels": True, "node_size": 4000, "node_color": "blue", "font_size": 10, "font_color": 'white', "font_weight": "bold", "edge_color": "gray", "width": 1.5}

# Grafo Dirigido
style_dir = 'arc3, rad=0.1' if len(G_dir.edges()) > 0 else None
nx.draw(G_dir, pos_dir, ax=ax1, arrowsize=25, connectionstyle=style_dir, **node_params)
ax1.set_title("Estructura Dirigida (DAG)", fontsize=15)

# Grafo No Dirigido
nx.draw(G_undir, pos_undir, ax=ax2, **node_params)
ax2.set_title("Estructura No Dirigida (Esqueleto)", fontsize=15)

fig.suptitle("Algoritmo PC: Comparativa de Estructura Dirigida vs No Dirigida", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

El resumen de las relaciones de causalidad dirigidas de nuestro dataset son:

- **MktRF**: Tiene relación de causalidad significativa con las variables **Cnsmr, HiTec y Other**.
- **SMB**:Tiene relación de causalidad significativa con la variable **HiTec**.
- **HML**: No tiene relaciones de causalidad.
- **RF**:Tiene relación de causalidad significativa con la variable **Mom**.
- **Mom**: No tiene relaciones de causalidad.
- **Cnsmr**: Tiene relación de causalidad significativa con las variables **Mkt-Rf, HiTec y Other**.
- **Manuf**: Tiene relación de causalidad significativa con la variable **HiTec**.
- **HiTec**: Tiene relación de causalidad significativa con las variables **Mkt-Rf y Cnsmr**.
- **Hlth**: Tiene relación de causalidad significativa con las variables **Mkt-Rf, Cnsmr, Manuf, HiTec y Other**.
- **Other**: Tiene relación de causalidad significativa con la variable **Manuf**.

# 4. Comparación de los grafos ante distintos factores

Se generan funciones para llamar a los gráficos de matrices direccionales y grafos acíclicos dirigidos (DAG) en cada uno de los casos.

In [ ]:
def visualizar_matrices_direccionales(modelos, n_filas, n_cols, titulos_subplots, titulo_general, columnas):
    """
    Dibuja una cuadrícula de matrices direccionales (heatmaps) para múltiples modelos causales.
    """
    # Ajustamos el tamaño de la figura dinámicamente según la cantidad de columnas y filas
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(8 * n_cols, 7 * n_filas))

    # Aseguramos que 'axes' sea siempre una lista plana
    if n_filas * n_cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # Iteramos sobre los ejes disponibles
    for i, ax in enumerate(axes):
        # Si aún tenemos modelos por dibujar
        if i < len(modelos):
            # 1. Extraer la matriz dirigida
            est_adj = nx.to_numpy_array(modelos[i].to_directed(), nodelist=list(columnas), weight=None)

            # 2. Dibujar el heatmap
            im = ax.imshow(est_adj)
            ax.set_title(titulos_subplots[i], fontsize=14, fontweight='bold')

            # 3. Configurar los nombres en los ejes
            ax.set_xticks(range(len(columnas)))
            ax.set_xticklabels(columnas, rotation=45)
            ax.set_yticks(range(len(columnas)))
            ax.set_yticklabels(columnas)
        else:
            ax.axis('off')

    plt.tight_layout()
    fig.suptitle(titulo_general, fontsize=18)

    margen_superior = 0.85 if n_filas == 1 else 0.92
    plt.subplots_adjust(top=margen_superior)
    plt.show()

In [ ]:
def visualizar_dags(modelos, mapping, titulos_subplots, colores, n_filas, n_cols, titulo_general, tipos=None):
    """
    Dibuja una cuadrícula de Grafos Acíclicos Dirigidos (DAGs) con nodos espaciados.
    """
    # Configuración dinámica de la figura
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(8 * n_cols, 8 * n_filas))

    if n_filas * n_cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # Parámetros estéticos base (MÁS SEPARACIÓN)
    node_params = {"with_labels": True, "node_size": 3000, "font_size": 10, "font_color": 'white', "font_weight": "bold", "edge_color": "gray", "width": 1.5, "arrowsize": 20}

    # Bucle para dibujar cada modelo
    for i, ax in enumerate(axes):
        if i < len(modelos):
            G = nx.DiGraph(modelos[i].to_directed())
            G = nx.relabel_nodes(G, mapping)
            G.remove_edges_from(list(nx.selfloop_edges(G)))

            # Comprobamos si el modelo es HC o PC
            es_hc = False
            if tipos is not None and i < len(tipos):
                if tipos[i].upper() == 'HC':
                    es_hc = True
            elif "HC" in titulos_subplots[i].upper() or "HILL" in titulos_subplots[i].upper():
                es_hc = True

            if es_hc:
                pos = nx.circular_layout(G, scale=1.5)
                style = 'arc3, rad=0.15' if len(G.edges()) > 0 else None
            else:
                pos = nx.spring_layout(G, k=1, iterations=50, seed=42)
                style = 'arc3, rad=0.05' if len(G.edges()) > 0 else None

            # Asignar color
            color_actual = colores[i] if i < len(colores) else "#3498db"

            # Dibujar el DAG
            nx.draw(G, pos, ax=ax, connectionstyle=style, node_color=color_actual, **node_params)
            ax.margins(0.20)
            ax.set_title(titulos_subplots[i], fontsize=16, fontweight='bold')

            if len(G.edges()) == 0:
                ax.text(0.5, 0.5, "Sin relaciones detectadas",
                        ha='center', transform=ax.transAxes, color='darkred', fontsize=12)
        else:
            ax.axis('off')

    margen_superior = 1.02 if n_filas == 1 else 0.95
    fig.suptitle(titulo_general, fontsize=20, y=margen_superior)
    plt.tight_layout()
    plt.show()

## 4.1. Según el periodo temporal

Dividimos los datos en periodos temporales diferentes. El objetivo es comprobar si las relaciones de causalidad se mantienen o modifican según el periodo de estudio.

Se considerarán tres periodos:
- 2007-2009
- 2010-2019
- 2020-2022

Se aprecia que en épocas de **crisis** sistemático como 2007-2009 y 2020-2022, **existen muchas menos relaciones de causalidad**. Como todo cae a la vez, el modelo no encuentra suficiente independencia condicional para decir que una variable específica causa el movimiento de otra. **Todo está siendo arrastrado por factores externos**, por lo que las flechas de causalidad interna desaparecen.

In [ ]:
# Datos del periodo 2007-2009
df_periodo_1 = df.loc['2007-01-01':'2009-12-31']
est_PC_periodo_1 = PC(data=df_periodo_1)
estimated_model_PC_periodo_1 = est_PC_periodo_1.estimate(variant="stable", max_cond_vars=2)

# Datos del periodo 2010-2019
df_periodo_2 = df.loc['2010-01-01':'2019-12-31']
est_PC_periodo_2 = PC(data=df_periodo_2)
estimated_model_PC_periodo_2 = est_PC_periodo_2.estimate(variant="stable", max_cond_vars=2)

# Datos del periodo 2020-2022
df_periodo_3 = df.loc['2020-01-01': '2022-12-31']
est_PC_periodo_3 = PC(data=df_periodo_3)
estimated_model_PC_periodo_3 = est_PC_periodo_3.estimate(variant="stable", max_cond_vars=2)

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelos_periodos = [estimated_model_PC_periodo_1, estimated_model_PC_periodo_2, estimated_model_PC_periodo_3]
titulos_periodos = ["Escenario 2007-2009", "Escenario 2010-2019", "Escenario 2020-2022"]
titulo_main_periodos = "Comparativa de Matrices Direccionales (3 Escenarios entre 2007 y 2022)"

visualizar_matrices_direccionales(modelos=modelos_periodos, n_filas=1, n_cols=3, titulos_subplots=titulos_periodos, titulo_general=titulo_main_periodos, columnas=df.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mapping_periodos = {i: col for i, col in enumerate(df.columns)}
colores_periodos = ["red", "green", "black"]

visualizar_dags(modelos=modelos_periodos,mapping=mapping_periodos, titulos_subplots=titulos_periodos, colores=colores_periodos, n_filas=1,n_cols=3, titulo_general=titulo_main_periodos)

## 4.2. Incluyendo y excluyendo LAGS

Para este caso, no vamos a utilizar todas las variables que tenemos disponibles porque corremos el riesgo de que la visualización no sea buena. Es por ello, que vamos a utilizar los factores Fama-French desde el año 2022:

- **Mkt-Rf**
- **SMB**
- **HML**
- **RF**
- **MOM**

Existe unicamente una **relación de causalidad entre una variable y su lag en el factor RF**, que encima es bidireccional. En el resto de variables no hay una relación de causalidad significativa con respecto a su lag.

In [ ]:
# Filtramos los datos desde 2022 y los factores Fama French
df_fama_french = df[['MktRF', 'SMB', 'HML', 'RF', 'Mom']].loc['2022-01-01':]

# Obtenemos los datos shift(1)
df_fama_french_shift = df_fama_french.shift(1)

# Concatenamos los dataframes
df_fama_french = pd.concat([df_fama_french, df_fama_french_shift], axis = 1).dropna()
df_fama_french.columns = ['MktRF', 'SMB', 'HML', 'RF', 'Mom', 'MktRF_shift', 'SMB_shift', 'HML_shift', 'RF_shift', 'Mom_shift']

# Entrenamos el modelo
est_PC_lag = PC(data=df_fama_french)
estimated_model_PC_lag = est_PC_lag.estimate(variant="stable", max_cond_vars=2)

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelo_lag = [estimated_model_PC_lag]
titulo_lag = ["Matriz direccional con factores Fama-French (lag)"]
titulo_main_lag = ""

visualizar_matrices_direccionales(modelos=modelo_lag, n_filas=1, n_cols=1, titulos_subplots=titulo_lag, titulo_general=titulo_main_lag, columnas=df_fama_french.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mi_mapping_lag = {i: col for i, col in enumerate(df_fama_french.columns)}
colores_lag = ["blue"]

visualizar_dags(modelos=modelo_lag, mapping=mi_mapping_lag, titulos_subplots=titulo_lag, colores=colores_lag, n_filas=1,n_cols=1, titulo_general=titulo_main_lag)

## 4.3. Según el método de ajuste

El **modelo PC es mucho más exigente** a la hora de considerar las relaciones entre variables como una causalidad significativa, por lo menos **en el ejemplo** que se muestra a continuación.

In [ ]:
# Entrenamos dos modelos (PC y HC) con datos desde 2022
est_PC_ajustes = PC(data=df.loc['2022-01-01':])
est_HC_ajustes = HillClimbSearch(data=df.loc['2022-01-01':])

estimated_model_ajuste_PC = est_PC_ajustes.estimate(variant="stable", max_cond_vars=2)
estimated_model_ajuste_HC = est_HC_ajustes.estimate()

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelos_ajustes = [estimated_model_ajuste_PC, estimated_model_ajuste_HC]
titulos_ajustes = ["Modelo PC", "Modelo Hill Climbing"]
titulo_main_ajuste = "Comparativa de Matrices Direccionales (2 métodos de ajuste)"

visualizar_matrices_direccionales(modelos=modelos_ajustes, n_filas=1, n_cols=2, titulos_subplots=titulos_ajustes, titulo_general=titulo_main_ajuste, columnas=df.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mapping_ajustes = {i: col for i, col in enumerate(df.columns)}
colores_ajustes = ["blue", "black"]

visualizar_dags(modelos=modelos_ajustes, mapping=mapping_ajustes, titulos_subplots=titulos_ajustes, colores=colores_ajustes, n_filas=1,n_cols=2, titulo_general=titulo_main_ajuste)

## 4.4. Según los parámetros del modelo

### Según el parámetro variant

No se aprecian apenas diferencias entre las diferentes variantes. En el caso variant = parallel considera como significativa la relación causal $Manuf → Other$ mientras que en el resto de casos no.

In [ ]:
# Según el parámetro variant
est_PC_variant = PC(data=df.loc['2022-01-01':])
estimated_model_PC_variant_1 = est_PC_variant.estimate(variant="stable", max_cond_vars=2)
estimated_model_PC_variant_2 = est_PC_variant.estimate(variant="orig", max_cond_vars=2)
estimated_model_PC_variant_3 = est_PC_variant.estimate(variant="parallel", max_cond_vars=2)

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelos_variants = [estimated_model_PC_variant_1, estimated_model_PC_variant_2, estimated_model_PC_variant_3]
titulos_variants = ["Variant = stable", "Variant = orig", "Variant = parallel"]
titulo_main_variants = "Comparativa de Matrices Direccionales (3 variantes)"

visualizar_matrices_direccionales(modelos=modelos_variants, n_filas=1, n_cols=3, titulos_subplots=titulos_variants, titulo_general=titulo_main_variants, columnas=df.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mapping_variants = {i: col for i, col in enumerate(df.columns)}
colores_variants = ["red", "green", "black"]

visualizar_dags(modelos=modelos_variants,mapping=mapping_variants, titulos_subplots=titulos_variants, colores=colores_variants, n_filas=1,n_cols=3, titulo_general=titulo_main_variants)

### Según el parámetro significance_level

Se aprecia que conforme sea menor el nivel de significancia (p-value) se considera que existe una menor cantidad de relaciones causales significativas. Esto es así porque para considerar una relación causal como significativa, el p-value de la relación debe ser menor para ser considerada así.

In [ ]:
# Según el parámetro significance_level
est_PC_SL = PC(data=df.loc['2022-01-01':])
estimated_model_PC_SL_1 = est_PC_SL.estimate(variant="stable", max_cond_vars=2, significance_level=0.1)
estimated_model_PC_SL_2 = est_PC_SL.estimate(variant="stable", max_cond_vars=2, significance_level=0.05)
estimated_model_PC_SL_3 = est_PC_SL.estimate(variant="stable", max_cond_vars=2, significance_level=0.01)

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelos_SL = [estimated_model_PC_SL_1, estimated_model_PC_SL_2, estimated_model_PC_SL_3]
titulos_SL = ["Significance level = 0.1", "Significance level = 0.05", "Significance level = 0.01"]
titulo_main_SL = "Comparativa de Matrices Direccionales (3 niveles de significancia)"

visualizar_matrices_direccionales(modelos=modelos_SL, n_filas=1, n_cols=3, titulos_subplots=titulos_SL, titulo_general=titulo_main_SL, columnas=df.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mapping_SL = {i: col for i, col in enumerate(df.columns)}
colores_SL = ["red", "green", "black"]

visualizar_dags(modelos=modelos_SL,mapping=mapping_SL, titulos_subplots=titulos_SL, colores=colores_SL, n_filas=1,n_cols=3, titulo_general=titulo_main_SL)

### Según el parámetro max_cond_vars

No se aprecia ninguna diferencia entre las relaciones de causalidad considerando 2, 3 y 4 variables condicionales para estimar relaciones de causalidad significativas.

In [ ]:
# Según el parámetro max_cond_vars
est_PC_MCV = PC(data=df.loc['2022-01-01':])
estimated_model_PC_MCV_1 = est_PC_MCV.estimate(variant="stable", max_cond_vars=2)
estimated_model_PC_MCV_2 = est_PC_MCV.estimate(variant="stable", max_cond_vars=3)
estimated_model_PC_MCV_3 = est_PC_MCV.estimate(variant="stable", max_cond_vars=4)

In [ ]:
# Preparamos las listas (modelos y títulos de cada subgráfico)
modelos_MCV = [estimated_model_PC_MCV_1, estimated_model_PC_MCV_2, estimated_model_PC_MCV_3]
titulos_MCV = ["Max_cond_vars = 2", "Max_cond_vars = 3", "Max_cond_vars = 4"]
titulo_main_MCV = "Comparativa de Matrices Direccionales (número de variables condicionadas)"

visualizar_matrices_direccionales(modelos=modelos_MCV, n_filas=1, n_cols=3, titulos_subplots=titulos_MCV, titulo_general=titulo_main_MCV, columnas=df.columns)

In [ ]:
# Definimos el resto de listas de parámetors
mapping_MCV = {i: col for i, col in enumerate(df.columns)}
colores_MCV = ["red", "green", "black"]

visualizar_dags(modelos=modelos_MCV,mapping=mapping_MCV, titulos_subplots=titulos_MCV, colores=colores_MCV, n_filas=1,n_cols=3, titulo_general=titulo_main_MCV)

## 4.5. Según la semilla

No se aprecian diferencias en las relaciones de causalidad con diferentes semillas en este caso. Es un ejemplo muy simple, para comprobar si dependiendo de las semillas se obtienen resultados diferentes habría que estudiar un número muy grande de semillas diferentes.

In [ ]:
# Entrenamos el modelo PC con datos desde 2022
est_PC_seed = PC(data=df.loc['2022-01-01':])
estimated_model_PC_seed = est_PC_seed.estimate(variant="stable", max_cond_vars=2)

In [ ]:
# Definimos el mapeo de nombres de las columnas
mapping_seed = {i: col for i, col in enumerate(df.columns)}

# Grafo Dirigido (Con flechas) para las tres semillas
G_dir_seed1 = nx.DiGraph(estimated_model_PC_seed.to_directed())
G_dir_seed1 = nx.relabel_nodes(G_dir_seed1, mapping_seed)
pos_dir_seed1 = nx.spring_layout(G_dir_seed1, k=1/np.sqrt(10), seed=42)

G_dir_seed2 = nx.DiGraph(estimated_model_PC_seed.to_directed())
G_dir_seed2 = nx.relabel_nodes(G_dir_seed2, mapping_seed)
pos_dir_seed2 = nx.spring_layout(G_dir_seed2, k=1/np.sqrt(10), seed=4353)

G_dir_seed3 = nx.DiGraph(estimated_model_PC_seed.to_directed())
G_dir_seed3 = nx.relabel_nodes(G_dir_seed3, mapping_seed)
pos_dir_seed3 = nx.spring_layout(G_dir_seed3, k=1/np.sqrt(10), seed=2)

# --- 3. Configuración de la figura (1 fila, 3 columnas) ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

# Parámetros estéticos comunes (¡SIN el node_color!)
node_params = {
    "with_labels": True,
    "node_size": 4000,
    "font_size": 10,
    "font_color": 'white',
    "font_weight": "bold",
    "edge_color": "gray",
    "width": 1.5
}

# --- 4. Dibujar Grafo Dirigido 1 (Azul) ---
style_dir_seed1 = 'arc3, rad=0.1' if len(G_dir_seed1.edges()) > 0 else None
nx.draw(G_dir_seed1, pos_dir_seed1, ax=ax1, arrowsize=25, connectionstyle=style_dir_seed1,
        node_color="blue", **node_params)
ax1.set_title("Estructura Dirigida (DAG): Caso seed = 42", fontsize=15)

# --- 5. Dibujar Grafo Dirigido 2 (Verde) ---
style_dir_seed2 = 'arc3, rad=0.1' if len(G_dir_seed2.edges()) > 0 else None
nx.draw(G_dir_seed2, pos_dir_seed2, ax=ax2, arrowsize=25, connectionstyle=style_dir_seed2,
        node_color="green", **node_params)
ax2.set_title("Estructura Dirigida (DAG): Caso seed = 4353", fontsize=15)

# --- 6. Dibujar Grafo Dirigido 3 (Rojo) ---
style_dir_seed3 = 'arc3, rad=0.1' if len(G_dir_seed3.edges()) > 0 else None
nx.draw(G_dir_seed3, pos_dir_seed3, ax=ax3, arrowsize=25, connectionstyle=style_dir_seed3,
        node_color="red", **node_params)
ax3.set_title("Estructura Dirigida (DAG): Caso seed = 2", fontsize=15)

# --- 7. Título global y ajuste ---
fig.suptitle("Algoritmo PC: Comparativa de Estructura Dirigida con diferentes semillas", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 4.6. Según el tipo de visualización

En este apartado se muestran los tres principales gráficos para mostrar las relaciones de causalidad:

- **Grafos acíclicos dirigidos**: Muestran las relaciones de causalidad entre cada par de variables en el espacio conectados por flechas que indican la dirección de la relación de causalidad.

- **Grafos no dirigidos**: Muestran los casos en los que existe una relación de causalidad entre cada par de variables sin importar la dirección.

- **Matrices direccionales**: Muestran para cada fila $i$ y columna $j$ la existencia o no de una relación causal significativa entre la variable de la fila $i$ y la variable de la columna $j$.

In [ ]:
def resumen_visual_modelo(modelo, columnas, titulo_general="Análisis Completo del Modelo Causal"):
    """
    Toma un modelo causal estimado y genera una figura con 3 subplots:
    1. DAG (Causalidad Dirigida)
    2. Esqueleto (Estructura No Dirigida)
    3. Matriz Direccional (Heatmap)
    """
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    mapping = {i: col for i, col in enumerate(columnas)}

    # Preparar Grafos (Dirigido y No Dirigido)
    G_dir = nx.DiGraph(modelo.to_directed())
    G_dir = nx.relabel_nodes(G_dir, mapping)
    G_dir.remove_edges_from(list(nx.selfloop_edges(G_dir)))

    G_undir = nx.Graph(modelo.to_undirected())
    G_undir = nx.relabel_nodes(G_undir, mapping)
    G_undir.remove_edges_from(list(nx.selfloop_edges(G_undir)))

    # Calculamos Posiciones (Layout)
    try:
        pos = nx.kamada_kawai_layout(G_dir)
    except:
        pos = nx.circular_layout(G_dir, scale=1.5)

    node_params = {"with_labels": True, "node_size": 3000, "font_size": 10, "font_color": 'white', "font_weight": "bold", "edge_color": "gray", "width": 1.5}

    # -Dibujamos el DAG
    style_dir = 'arc3, rad=0.1' if len(G_dir.edges()) > 0 else None
    nx.draw(G_dir, pos, ax=axes[0], arrowsize=20, connectionstyle=style_dir,
            node_color="#3498db", **node_params) # Azul
    axes[0].set_title("1. Grafo Causal Dirigido (DAG)", fontsize=15, fontweight='bold')
    axes[0].margins(0.20)
    if len(G_dir.edges()) == 0:
        axes[0].text(0.5, 0.5, "Sin causalidad", ha='center', transform=axes[0].transAxes, color='red')

    # Dibujamos el grafo no dirigido
    nx.draw(G_undir, pos, ax=axes[1], node_color="#95a5a6", **node_params) # Gris neutro
    axes[1].set_title("2. Esqueleto (grafo no dirigido)", fontsize=15, fontweight='bold')
    axes[1].margins(0.20)

    # Dibujamos la matriz direccional
    # Extraemos la matriz asegurando el orden correcto de las columnas
    adj_matrix = nx.to_numpy_array(modelo.to_directed(), nodelist=list(columnas), weight=None)
    im = axes[2].imshow(adj_matrix)
    axes[2].set_title("3. Matriz Direccional", fontsize=15, fontweight='bold')
    axes[2].set_xticks(range(len(columnas)))
    axes[2].set_xticklabels(columnas, rotation=45)
    axes[2].set_yticks(range(len(columnas)))
    axes[2].set_yticklabels(columnas)

    fig.suptitle(titulo_general, fontsize=20, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Calculamos el modelo PC con datos desde 2022
est_PC_seed = PC(data=df.loc['2022-01-01':])
estimated_model_PC_seed = est_PC_seed.estimate(variant="stable", max_cond_vars=2)

In [ ]:
# Obtenemos las tres formas de visualización utilizadas en el notebook
resumen_visual_modelo(modelo=estimated_model_PC_seed, columnas=df.columns, titulo_general="Análisis Visual: Algoritmo PC (Datos desde 2022)")

# 5. Análisis del factor mirage (Extra)

En el mundo financiero tradicional, los investigadores usan regresiones estadísticas para encontrar factores que predicen retornos (como el factor Tamaño, Valor, Momentum). El problema es que estas herramientas estadísticas tradicionales **premian cualquier cosa que aumente el $R^2$, sin importar si la relación tiene sentido lógico**.

El **Factor Mirage** es un modelo que parece estadísticamente perfecto y aceptable en un backtest, pero que en realidad es **estructuralmente inválido porque confunde correlación con causalidad**. López de Prado explica que este espejismo ocurre principalmente por el error del **sesgo de colisionador** (Collider Bias).

- **¿Qué es un Colisionador?** En el caso en el que dispongamos de tres variables, si la variable A causa la C ($A → C$) y la variable B también causa la C ($B → C$). En este caso, C es un colisionador.
- **El Error**: Si al hacer un modelo matemático decidimos controlar o incluir a la variable C, mágicamente aparecerá una correlación matemática falsa entre A y B.
- **El Peligro**: Esto puede hacer que el **signo de la rentabilidad se invierta**. Un inversor podría ver el espejismo, pensar que el Factor A causa retornos positivos, comprar esa acción y perder dinero en la vida real porque la relación era falsa y no se puede monetizar.

En los **apartados 4.3 y 4.4** podemos sospechar de una existencia de una relación espuria tipo colisionador gracias a las matrices direccionales y grafos DAG. Las tres variables que intervienen son:

- **Other**
- **Manuf**
- **Mkt-RF**

Las relaciones de causalidad entre estas variables son:

**Other**     $\rightarrow$      **Manuf**

     $\searrow$            $\swarrow$

           **MktRF**

Entrenamos los siguientes dos modelos de regresión lineal para comprobar el efecto del colisionador (Mkt-RF) y la existencia de una relación espuria.

- Modelo sin colisionador: Manuf = $\beta_0$ + $\beta_1$*Other + $ϵ$

- Modelo con colisionador: Manuf = $\beta_0$ + $\beta_1$*Other + $\beta_2$*Mkt-RF + $ϵ$

Comprobamos para cada uno de los modelos:

- **$\beta_1$** asociada a la variable Other y como cambia de signo al incorporar en el modelo el colisionador Mkt-RF.
- **$R^2$** y como aumenta al incluir el colisionador, generando una **relación espuria**.

In [ ]:
# Definición de variables
variable_objetivo = 'Manuf'
causa_verdadera = 'Other'
colisionador = 'MktRF'

# Entrenamos el modelo sin colisionador (Mkt-RF)
X_correcta = sm.add_constant(datos[[causa_verdadera]])
modelo_correcto = sm.OLS(datos[variable_objetivo], X_correcta).fit()
# Obtenemos los parámetros clave a investigar del modelo beta_1 y R^2 de la variable Other
beta_correcto = modelo_correcto.params[causa_verdadera]
r2_correcto = modelo_correcto.rsquared_adj

# Entrenamos el modelo con colisionador (Mkt-RF)
X_mirage = sm.add_constant(datos[[causa_verdadera, colisionador]])
modelo_mirage = sm.OLS(datos[variable_objetivo], X_mirage).fit()
# Obtenemos los parámetros clave a investigar del modelo beta_1 y R^2 de la variable Other
beta_mirage = modelo_mirage.params[causa_verdadera]
r2_mirage = modelo_mirage.rsquared_adj

# Generamos la tabla comparativa entre modelos
tabla_mirage = pd.DataFrame({
    'Factor a Explicar': [variable_objetivo],
    'Causa Real': [causa_verdadera],
    'Beta Correcto': [round(beta_correcto, 4)],
    'R2 Correcto': [round(r2_correcto, 4)],
    'Colisionador': [colisionador],
    'Beta Sesgado (Mirage)': [round(beta_mirage, 4)],
    'R2 Sesgado (Mirage)': [round(r2_mirage, 4)]})

# Mostrar la tabla en el Notebook
display(tabla_mirage)

Observando únicamente el **$R^2$, sube de 0.77 a 0.8947 incluyendo en el modelo el colisionador (Mkt-RF)** y concluiríamos erróneamente que incluirla mejora el modelo. Este refleja lo que es el 'Factor Mirage': un modelo aparentemente más robusto y que captura una mayor varianza de los datos pero que en realidad es fruto de una relación espuria que distorsiona los resultados.

Gracias a nuestro DAG previo, sabemos que el factor Mkt-RF actúa como un colisionador en este escenario. Al incluirlo en la regresión, hemos introducido una asociación espuria. Esto provoca que el coeficiente **$\beta_1$ de la variable Other se distorsione, bajando de 0.7928 a -0.0235**. De este modo, estaríamos subestimando la exposición real al riesgo de mercado, lo que podría llevar a una mala asignación de capital.